# Algorithm Forge - Iterative Self-Improvement Loop (OLLAMA-ONLY)

**Vollstaendig kostenlos.** Alles laeuft auf der gratis Colab T4 GPU - kein OpenAI, kein API-Key noetig. Nur HF_TOKEN fuer Upload deiner Resultate.

**LLM:** Qwen2.5-Coder-14B (~9GB, faehiger als 7B, passt auf T4)

**Was passiert:**
1. Ollama in Colab installieren + Qwen2.5-Coder-14B pullen
2. Iter 0: 14B generiert Mutationen fuer sort + matmul
3. Fine-Tuning: Qwen2.5-Coder-7B mit Unsloth auf den Mutationen (kleineres Base-Modell weil Fine-Tuning mehr VRAM braucht)
4. GGUF Export -> Ollama registrieren als `forge-mutator-v1`
5. Iter 1: der fine-tuned Mutator generiert neue Mutationen -> aufkummuliertes Dataset -> v2
6. A/B Vergleich pro Iteration

**VRAM-Management:** Ollama wird vor jedem Fine-Tuning gestoppt (Modell-Unload), nach Fine-Tuning neu gestartet.

**Resilient:** alles in Google Drive. Wenn Colab disconnected -> Notebook neu oeffnen, Run All, macht da weiter wo's aufgehoert hat.

## 1. Konfiguration

In [ ]:
# === HAUPT-KNOEPFE ===
ITERATIONS = 2              # Wie viele V1, V2, V3... Iterationen?
RUNS_PER_BENCHMARK = 4      # Wie viele seeded Runs je Benchmark je Iteration?
GENERATIONS_PER_RUN = 20    # Mutationen pro Run
FT_EPOCHS = 2               # Fine-Tuning Epochen je Iteration

# === Bootstrap-LLM (lokal in Ollama, Iter 0) ===
BOOTSTRAP_OLLAMA_MODEL = "qwen2.5-coder:14b"   # ~9GB, T4-fit, faehigster Free-Coder
# Alternative wenn 14B zu langsam (~50% schneller): "qwen2.5-coder:7b"

# === Fine-Tuning Base (kleiner, damit VRAM fuer Training reicht) ===
FT_BASE = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"

# === Benchmarks ===
BENCHMARKS = ["sort", "matmul"]

# === HF Hub ===
HF_USERNAME = "Beko2210"
HF_DATASET_PREFIX = f"{HF_USERNAME}/algorithm-forge-mutations"
HF_MODEL_PREFIX = f"{HF_USERNAME}/algorithm-forge-mutator"

# === Repo ===
REPO_URL = "https://github.com/BEKO2210/Science_game-.git"
BRANCH = "claude/evolution-game-concept-yN3Tn"

# === Persistenz ===
DRIVE_WORK_DIR = "/content/drive/MyDrive/algorithm-forge"

print(f"Plan: {ITERATIONS} iterations x {RUNS_PER_BENCHMARK} runs x {len(BENCHMARKS)} benchmarks x {GENERATIONS_PER_RUN} gens")
print(f"Total LLM calls per iter: {RUNS_PER_BENCHMARK * len(BENCHMARKS) * GENERATIONS_PER_RUN}")
print(f"Bootstrap model: {BOOTSTRAP_OLLAMA_MODEL}")
print(f"FT base: {FT_BASE}")
print(f"COST: $0 (alles lokal auf T4)")
print(f"TIME: ~{ITERATIONS * 70} min")

## 2. Google Drive + Resume-Helpers

In [ ]:
import os, sys, json, subprocess, shutil, time, signal
from pathlib import Path
from google.colab import drive, userdata

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

WORK_DIR = Path(DRIVE_WORK_DIR)
for sub in ["runs", "datasets", "checkpoints", "reports", "status", "gguf"]:
    (WORK_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"Persistent work dir: {WORK_DIR}")
for p in sorted(WORK_DIR.iterdir()):
    print(f"  {p.name}/")

# Resume helpers (auch nochmal an Hauptfunktionen-Zelle definiert, hier defensive Doppelung)
def step_done(iteration: int, step: str) -> bool:
    return (WORK_DIR / "status" / f"iter{iteration}_{step}.done").exists()

def mark_step_done(iteration: int, step: str):
    (WORK_DIR / "status" / f"iter{iteration}_{step}.done").write_text(
        f"done at {time.strftime('%Y-%m-%d %H:%M:%S')}"
    )

def reset_step(iteration: int, step: str):
    f = WORK_DIR / "status" / f"iter{iteration}_{step}.done"
    if f.exists(): f.unlink()

print("\n=== Aktueller Fortschritt ===")
for status in sorted((WORK_DIR / "status").glob("*.done")):
    print(f"  OK {status.stem}")

## 3. Repo + Dependencies

In [ ]:
REPO_DIR = Path("/content/Science_game-")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--recursive", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

os.chdir(REPO_DIR)

# Drive symlinks
for sub in ["runs", "datasets", "checkpoints"]:
    local = REPO_DIR / sub
    target = WORK_DIR / sub
    if local.is_symlink() or local.exists():
        if local.is_symlink(): local.unlink()
        elif local.is_dir(): shutil.rmtree(local)
    local.symlink_to(target)

print("Commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

# uv sync - WICHTIG: kein --extra api-llm noetig, kein OpenAI!
subprocess.run(["pip", "install", "-qU", "uv"], check=True)
subprocess.run(["uv", "sync", "--extra", "dev"], check=True)
print("uv sync done.")

In [ ]:
# HF Token (nur fuer Upload, kein anderer API-Key)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF Hub authentifiziert.")

import glob

if shutil.which("ollama") is None:
    print("Installing Ollama from GitHub releases...")

    # Mehrere URLs probieren — Ollama hat schon mal Download-Pfade umgebaut
    urls = [
        "https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tgz",
        "https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64",
    ]

    downloaded = None
    for url in urls:
        target = "/tmp/ollama.tgz" if url.endswith(".tgz") else "/tmp/ollama-bin"
        print(f"  Trying {url}...")
        r = subprocess.run(
            ["curl", "-L", "-o", target, "-w", "HTTP %{http_code}\\n", url],
            capture_output=True, text=True,
        )
        print(f"    -> {r.stdout.strip()}")
        if os.path.exists(target) and os.path.getsize(target) > 1_000_000:
            downloaded = target
            print(f"    OK ({os.path.getsize(target) // 1024 // 1024} MB)")
            break

    if downloaded is None:
        raise RuntimeError("Kein URL hat funktioniert. Schau in den Output oben.")

    if downloaded.endswith(".tgz"):
        subprocess.run(["tar", "-xzf", downloaded, "-C", "/usr/local"], check=True)
    else:
        subprocess.run(["install", "-m", "0755", downloaded, "/usr/local/bin/ollama"], check=True)

    if shutil.which("ollama") is None:
        candidates = glob.glob("/usr/local/**/ollama", recursive=True)
        if candidates:
            os.environ["PATH"] = f"{Path(candidates[0]).parent}:{os.environ.get('PATH','')}"

    v = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print("\nInstalled:", v.stdout.strip() or v.stderr.strip())
else:
    v = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print("Ollama already installed:", v.stdout.strip())

In [ ]:
# Ollama installieren (idempotent)
if shutil.which("ollama") is None:
    print("Installing Ollama...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    print("Ollama installed.")
else:
    print("Ollama already installed:", subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Ollama-Modelle nach Drive ausziehen damit sie Disconnects ueberleben
OLLAMA_MODELS_DIR = WORK_DIR / "ollama_models"
OLLAMA_MODELS_DIR.mkdir(exist_ok=True)
os.environ["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
print(f"OLLAMA_MODELS = {OLLAMA_MODELS_DIR}")
print(f"  contents: {[p.name for p in OLLAMA_MODELS_DIR.iterdir()] or '(empty)'}")

def ollama_serve_start():
    """Start Ollama as a background process. Idempotent."""
    # Check if already running
    try:
        import httpx
        r = httpx.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            return
    except Exception:
        pass
    env = os.environ.copy()
    env["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
    subprocess.Popen(["ollama", "serve"], env=env,
                     stdout=open("/tmp/ollama.log", "a"),
                     stderr=subprocess.STDOUT)
    # Wait for service to come up
    import httpx
    for _ in range(30):
        try:
            if httpx.get("http://localhost:11434/api/tags", timeout=1).status_code == 200:
                print("  Ollama service ready.")
                return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama service didn't start within 30s")

def ollama_serve_stop():
    """Kill Ollama to free VRAM before fine-tuning."""
    subprocess.run(["pkill", "-9", "ollama"], check=False)
    time.sleep(2)
    print("  Ollama stopped (VRAM freed).")

ollama_serve_start()
print("Ollama service running.")

In [ ]:
# Bootstrap-Modell ziehen (idempotent - wenn da, kein Re-Download)
import httpx

def ollama_has_model(name: str) -> bool:
    try:
        r = httpx.get("http://localhost:11434/api/tags", timeout=5)
        return name in [m.get("name", "") for m in r.json().get("models", [])]
    except Exception:
        return False

if not ollama_has_model(BOOTSTRAP_OLLAMA_MODEL):
    print(f"Pulling {BOOTSTRAP_OLLAMA_MODEL}... (one-time ~5-10 min download)")
    proc = subprocess.Popen(["ollama", "pull", BOOTSTRAP_OLLAMA_MODEL],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
    proc.wait()
else:
    print(f"{BOOTSTRAP_OLLAMA_MODEL} already pulled.")

print("\nAvailable Ollama models:")
subprocess.run(["ollama", "list"])

## 5. Unsloth (fuer Fine-Tuning)

In [ ]:
import importlib.util
if importlib.util.find_spec("unsloth") is None:
    subprocess.run(["pip", "install", "-qU", "unsloth"], check=True)
    subprocess.run(["pip", "install", "-qU",
                    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"], check=True)
print("Unsloth ready.")

## 6. Hauptfunktionen

Jede Step ist idempotent - bereits-erledigte Steps werden uebersprungen (Marker in Drive).

In [ ]:
from typing import Optional

# Helpers nochmal hier definieren - selbst-enthaltene Zelle (defensive)
def step_done(iteration: int, step: str) -> bool:
    return (WORK_DIR / "status" / f"iter{iteration}_{step}.done").exists()

def mark_step_done(iteration: int, step: str):
    (WORK_DIR / "status" / f"iter{iteration}_{step}.done").write_text(
        f"done at {time.strftime('%Y-%m-%d %H:%M:%S')}"
    )

def reset_step(iteration: int, step: str):
    f = WORK_DIR / "status" / f"iter{iteration}_{step}.done"
    if f.exists(): f.unlink()


def generate_runs(iteration: int, ollama_model: str):
    """Step A: generate seeded runs using an Ollama-hosted model."""
    if step_done(iteration, "runs"):
        print(f"[iter {iteration}] runs already generated, skipping.")
        return

    # Sicherstellen dass Ollama laeuft
    ollama_serve_start()
    if not ollama_has_model(ollama_model):
        raise RuntimeError(f"Ollama model {ollama_model!r} nicht da. Pull es zuerst: ollama pull {ollama_model}")

    iter_runs_root = Path(f"runs/iter{iteration}")
    iter_runs_root.mkdir(parents=True, exist_ok=True)

    for bench in BENCHMARKS:
        for seed_offset in range(RUNS_PER_BENCHMARK):
            seed = iteration * 1000 + seed_offset
            args = [
                "uv", "run", "science-game", "run", bench,
                "--provider", "ollama-qwen",
                "--model", ollama_model,
                "--generations", str(GENERATIONS_PER_RUN),
                "--seed", str(seed),
                "--temperature", "0.9",
                "--runs-root", str(iter_runs_root),
            ]
            print(f"  [iter {iteration}] {bench} seed={seed} ({ollama_model})...", end=" ", flush=True)
            t0 = time.time()
            proc = subprocess.run(args, capture_output=True, text=True)
            if proc.returncode != 0:
                print(f"FAIL ({proc.stderr[-200:]})")
                continue
            best = [l for l in proc.stdout.splitlines() if "Best fitness" in l]
            print(f"{time.time()-t0:.1f}s  {best[0] if best else ''}")

    mark_step_done(iteration, "runs")

In [ ]:
def aggregate_dataset(iteration: int) -> Path:
    """Step B: aggregate ALL runs from iter 0..N into ONE cumulative dataset."""
    out_path = WORK_DIR / "datasets" / f"mutator-v{iteration + 1}.jsonl"
    if step_done(iteration, "dataset") and out_path.exists():
        n = sum(1 for _ in out_path.open())
        print(f"[iter {iteration}] dataset already built ({n} examples).")
        return out_path

    cumulative_runs = Path("runs/_cumulative")
    if cumulative_runs.exists(): shutil.rmtree(cumulative_runs)
    cumulative_runs.mkdir(parents=True)
    for i in range(iteration + 1):
        iter_dir = Path(f"runs/iter{i}")
        if not iter_dir.exists(): continue
        for run in iter_dir.iterdir():
            if run.is_dir():
                dest = cumulative_runs / f"i{i}_{run.name}"
                if not dest.exists():
                    shutil.copytree(run, dest)
    print(f"  Cumulative runs (iter 0..{iteration}): {len(list(cumulative_runs.iterdir()))}")

    subprocess.run([
        "uv", "run", "science-game", "build-mutator-dataset",
        "--runs-root", str(cumulative_runs),
        "--out", str(out_path),
        "--mode", "sft", "--min-delta", "0.0001",
    ], check=True)
    n = sum(1 for _ in out_path.open())
    print(f"  Dataset: {n} examples -> {out_path}")
    if n < 3:
        raise RuntimeError(
            f"Dataset zu klein ({n}). Das Bootstrap-Modell hat keine akzeptierten "
            f"Mutationen produziert. Versuche groesseres GENERATIONS_PER_RUN, oder ein "
            f"staerkeres Modell wie qwen2.5-coder:14b (Aktuelles: {BOOTSTRAP_OLLAMA_MODEL})."
        )
    mark_step_done(iteration, "dataset")
    return out_path

In [ ]:
def push_dataset(iteration: int, dataset_path: Path) -> str:
    """Step C: push dataset to HF Hub."""
    repo_id = f"{HF_DATASET_PREFIX}-v{iteration + 1}"
    if step_done(iteration, "push_dataset"):
        return repo_id
    from huggingface_hub import HfApi, create_repo
    create_repo(repo_id, repo_type="dataset", exist_ok=True, private=False)
    HfApi().upload_file(
        path_or_fileobj=str(dataset_path), path_in_repo="data.jsonl",
        repo_id=repo_id, repo_type="dataset",
    )
    print(f"  Dataset hochgeladen: https://huggingface.co/datasets/{repo_id}")
    mark_step_done(iteration, "push_dataset")
    return repo_id

In [ ]:
def fine_tune(iteration: int, dataset_repo: str) -> Path:
    """Step D: Stop Ollama (VRAM!), fine-tune Qwen2.5-Coder-7B mit Unsloth."""
    ckpt = WORK_DIR / "checkpoints" / f"mutator-v{iteration + 1}"
    if step_done(iteration, "finetune") and ckpt.exists():
        print(f"[iter {iteration}] checkpoint already exists: {ckpt}")
        return ckpt

    print("  Stopping Ollama to free VRAM for Unsloth...")
    ollama_serve_stop()

    from unsloth import FastLanguageModel
    from datasets import load_dataset
    from unsloth.chat_templates import standardize_sharegpt
    from trl import SFTTrainer
    from transformers import TrainingArguments
    import torch

    print(f"  [iter {iteration}] Loading {FT_BASE}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=FT_BASE,
        max_seq_length=4096, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=42,
    )

    print(f"  Loading dataset {dataset_repo}...")
    ds = load_dataset(dataset_repo, split="train")
    ds = standardize_sharegpt(ds)
    print(f"  Training on {len(ds)} examples, {FT_EPOCHS} epochs...")

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        dataset_text_field="text", max_seq_length=4096,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=5, num_train_epochs=FT_EPOCHS, learning_rate=2e-4,
            logging_steps=5, optim="adamw_8bit", seed=42,
            output_dir=f"/content/ft_out_v{iteration + 1}",
            report_to="none",
        ),
    )
    trainer.train()

    ckpt.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(ckpt))
    tokenizer.save_pretrained(str(ckpt))
    print(f"  Saved LoRA checkpoint -> {ckpt}")

    # GGUF Export fuer Ollama
    gguf_dir = WORK_DIR / "gguf" / f"mutator-v{iteration + 1}"
    gguf_dir.mkdir(parents=True, exist_ok=True)
    print(f"  Exporting GGUF -> {gguf_dir} (5-10 min)...")
    model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method="q4_k_m")

    # Free VRAM
    del model, tokenizer, trainer; torch.cuda.empty_cache()

    mark_step_done(iteration, "finetune")
    return ckpt

In [ ]:
def register_in_ollama(iteration: int) -> str:
    """Step E: nimm das frisch exportierte GGUF + registriere es als Ollama-Modell.
    Restart Ollama-Service danach."""
    model_name = f"forge-mutator-v{iteration + 1}"
    if step_done(iteration, "ollama_register"):
        ollama_serve_start()
        return model_name

    gguf_dir = WORK_DIR / "gguf" / f"mutator-v{iteration + 1}"
    gguf_files = list(gguf_dir.glob("*.gguf"))
    if not gguf_files:
        raise RuntimeError(f"Kein GGUF in {gguf_dir} gefunden")
    gguf_path = gguf_files[0]

    modelfile_path = gguf_dir / "Modelfile"
    modelfile_path.write_text(f'''FROM {gguf_path}
PARAMETER temperature 0.8
PARAMETER num_ctx 8192
SYSTEM "You are an evolutionary code mutator. Given a Python program and a benchmark description, you produce an improved variant. Output ONLY a single fenced Python code block."
''')

    ollama_serve_start()
    print(f"  Registering {model_name} in Ollama from {gguf_path.name}...")
    proc = subprocess.run(["ollama", "create", model_name, "-f", str(modelfile_path)],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"ollama create failed:\n{proc.stderr}")
    print(f"  {model_name} registriert.")
    subprocess.run(["ollama", "list"])
    mark_step_done(iteration, "ollama_register")
    return model_name

In [ ]:
def push_lora_to_hub(iteration: int, ckpt: Path):
    """Step F: push LoRA adapter to HF Hub."""
    if step_done(iteration, "push_model"): return
    from huggingface_hub import HfApi, create_repo
    repo_id = f"{HF_MODEL_PREFIX}-v{iteration + 1}"
    create_repo(repo_id, repo_type="model", exist_ok=True, private=False)
    HfApi().upload_folder(
        folder_path=str(ckpt), repo_id=repo_id, repo_type="model",
        ignore_patterns=["*.bin", "global_step*"],
    )
    print(f"  LoRA hochgeladen: https://huggingface.co/{repo_id}")
    mark_step_done(iteration, "push_model")

In [ ]:
def ab_compare(iteration: int, finetuned_ollama_name: str):
    """Step G: A/B compare fine-tuned mutator vs base Qwen2.5-Coder."""
    report_path = WORK_DIR / "reports" / f"ab_iter{iteration}.json"
    if step_done(iteration, "ab") and report_path.exists():
        print(f"[iter {iteration}] A/B already done -> {report_path}")
        return json.loads(report_path.read_text())

    ollama_serve_start()
    if not ollama_has_model(finetuned_ollama_name):
        raise RuntimeError(f"Fine-tuned Ollama model {finetuned_ollama_name!r} nicht registriert")
    if not ollama_has_model(BOOTSTRAP_OLLAMA_MODEL):
        raise RuntimeError(f"Base model {BOOTSTRAP_OLLAMA_MODEL!r} nicht da")

    AB_SEEDS = [9000, 9001, 9002]; AB_GEN = 12; AB_BENCH = "sort"
    pairs = []
    for s in AB_SEEDS:
        base_dir = Path(f"runs/ab/iter{iteration}/seed{s}-base")
        ft_dir = Path(f"runs/ab/iter{iteration}/seed{s}-ft")

        def _run_via_cli(run_dir, model_name):
            args = [
                "uv", "run", "science-game", "run", AB_BENCH,
                "--provider", "ollama-qwen", "--model", model_name,
                "--generations", str(AB_GEN), "--seed", str(s),
                "--temperature", "0.9",
                "--runs-root", str(run_dir.parent),
            ]
            proc = subprocess.run(args, capture_output=True, text=True)
            # Find the actual created run-id directory
            created = sorted(run_dir.parent.glob(f"{AB_BENCH}-*"))[-1]
            # Parse best fitness from stdout
            for line in proc.stdout.splitlines():
                if "Best fitness" in line:
                    import re
                    m = re.search(r"Best fitness:\s*([0-9.e+-]+)", line)
                    if m: return float(m.group(1))
            return 0.0

        print(f"  [iter {iteration}] A/B seed {s}...", end=" ", flush=True)
        base_fit = _run_via_cli(base_dir, BOOTSTRAP_OLLAMA_MODEL)
        ft_fit = _run_via_cli(ft_dir, finetuned_ollama_name)
        delta = ft_fit - base_fit
        winner = "finetuned" if delta > 1e-9 else ("base" if delta < -1e-9 else "tie")
        print(f"base={base_fit:.6f}  ft={ft_fit:.6f}  -> {winner}")
        pairs.append({"seed": s, "base_fitness": base_fit, "ft_fitness": ft_fit,
                      "delta": delta, "winner": winner})

    report = {"iteration": iteration, "pairs": pairs,
              "avg_delta": sum(p["delta"] for p in pairs) / len(pairs),
              "wins_ft": sum(1 for p in pairs if p["winner"] == "finetuned"),
              "wins_base": sum(1 for p in pairs if p["winner"] == "base"),
              "ties": sum(1 for p in pairs if p["winner"] == "tie")}
    report_path.write_text(json.dumps(report, indent=2))
    print(f"  Report -> {report_path}")
    print(f"  Summary: ft wins {report['wins_ft']}/{len(pairs)}, avg delta {report['avg_delta']:+.6f}")
    mark_step_done(iteration, "ab")
    return report

## 7. Main loop

Pro Iteration: A) generate runs B) build dataset C) push dataset D) fine-tune + GGUF E) register in Ollama F) push LoRA G) A/B compare.

In [ ]:
all_reports = []
previous_ollama_model = None

for iteration in range(ITERATIONS):
    print(f"\n{'=' * 60}\nITERATION {iteration + 1}/{ITERATIONS}\n{'=' * 60}")

    # A) Generate runs
    if iteration == 0:
        gen_model = BOOTSTRAP_OLLAMA_MODEL
    else:
        gen_model = previous_ollama_model
    generate_runs(iteration, gen_model)

    # B-C) Aggregate + push dataset
    ds_path = aggregate_dataset(iteration)
    ds_repo = push_dataset(iteration, ds_path)

    # D) Fine-tune + GGUF export (this stops Ollama internally)
    ckpt = fine_tune(iteration, ds_repo)

    # E) Register GGUF in Ollama (restarts service)
    ollama_name = register_in_ollama(iteration)

    # F) Push LoRA adapter to HF Hub
    push_lora_to_hub(iteration, ckpt)

    # G) A/B compare
    report = ab_compare(iteration, ollama_name)
    all_reports.append(report)

    previous_ollama_model = ollama_name

print("\n=== ALL ITERATIONS DONE ===")

## 8. Final-Report & Plot

In [ ]:
all_report_files = sorted((WORK_DIR / "reports").glob("ab_iter*.json"),
                          key=lambda p: int(p.stem.replace("ab_iter", "")))
all_reports = [json.loads(p.read_text()) for p in all_report_files]

print(f"\n{'iter':<5} {'ft_wins':<10} {'base_wins':<10} {'ties':<6} {'avg_delta':<12}")
print("-" * 50)
for r in all_reports:
    print(f"{r['iteration']:<5} {r['wins_ft']:<10} {r['wins_base']:<10} {r['ties']:<6} {r['avg_delta']:+.6f}")

try:
    import matplotlib.pyplot as plt
    iters = [r['iteration'] for r in all_reports]
    deltas = [r['avg_delta'] for r in all_reports]
    plt.figure(figsize=(8, 4))
    plt.plot(iters, deltas, marker='o', linewidth=2)
    plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
    plt.xlabel('Iteration'); plt.ylabel('Avg fitness delta (ft - base)')
    plt.title('Self-improvement progress: fine-tuned mutator vs Qwen2.5-Coder')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(WORK_DIR / 'reports' / 'progress.png', dpi=100); plt.show()
    print(f"Plot saved -> {WORK_DIR / 'reports' / 'progress.png'}")
except Exception as e:
    print(f"Plot skipped: {e}")

## 9. Done!

Deine fine-tuned Modelle:
- In Ollama: `forge-mutator-v1`, `forge-mutator-v2`, ... (via `ollama list`)
- Auf HF Hub: `Beko2210/algorithm-forge-mutator-vN` (LoRA Adapter)
- Datasets: `Beko2210/algorithm-forge-mutations-vN`
- GGUFs in Drive: `MyDrive/algorithm-forge/gguf/`

**Wenn der Plot nach oben geht** (avg_delta steigt ueber iter): du hast es geschafft, eine eigene KI mit eigenen KI-Mutationsdaten zu trainieren die messbar besser wird.

Lokal weiterbenutzen:
```bash
# GGUF aus Drive runterladen, dann lokal:
ollama create forge-mutator-v2 -f Modelfile
uv run science-game run sort --provider ollama-qwen --model forge-mutator-v2 -g 30
```